# Neurotransmitter Probability Variance across Drosophila Neuropils

This projects aims to answer - how do neurotransmitter probability distributions vary across neuropils in Drosophila?
The datasets should be downloaded into the data directory following the instructions on GitHub. 

## Set up environment

In [1]:
%load_ext autoreload 
%autoreload 2

# Import external libraries
from IPython.core.display import HTML
import pyvista as pv
from dask.distributed import Client

# Import core python libraries
import os

# Import local scripts (brainz.py, pipeline.py, plotting.py, 
# preprocess.py, util.py)
from scripts import *

In [2]:
# Configure Dask Client using threads because pipeline is data transfer heavy.
allow_bytes = pipeline.get_ram_allowance()
client = Client(processes=False, 
                threads_per_worker=os.cpu_count(), 
                memory_limit=allow_bytes)
print(f"\nDask Dashboard at {client.dashboard_link}\n")

15.8 GB available on machine; allowing 7.9 GB

Dask Dashboard at http://192.168.0.209:8787/status



2026-05-29 09:04:25,185 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle b05a5b8a077948b12fdbb5b4531d3dbe initialized by task ('shuffle-transfer-b05a5b8a077948b12fdbb5b4531d3dbe', 0) executed on worker inproc://192.168.0.209/4232/4
2026-05-29 09:04:25,348 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle b05a5b8a077948b12fdbb5b4531d3dbe deactivated due to stimulus 'task-finished-1780002265.3459759'
2026-05-29 09:04:32,931 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- Unmanaged memory: 5.83 GiB -- Worker memory limit: 7.90 GiB
2026-05-29 09:04:37,928 - distributed.worker.memory - WARNING - Worker is at 97% memory usage. Pausing worker.  Process memory: 7.67 GiB -- Worker memory limit: 7.90 GiB
2026-05-29 09:04:44,231 - distributed.worker.memory - 

In [3]:
# Set up in-line 3D plot rendering
pv.set_jupyter_backend("client")
HTML("""
<style>
.output_svg {
    display: table-cell;
    text-align: center;
    vertical-align: middle;
}
</style>
""")

In [4]:
# Set globals
OUTDIR = os.path.join(os.path.dirname(__name__), "results", "notebook")
if not os.path.exists(OUTDIR): os.mkdir(OUTDIR)
MINSIZE = 30 # Minimum number of nodes in a cluster/community

In [5]:
preprocess.run() # ~10 mins first run

Notice: this may take about 10 minutes if this is the first time running preprocess.py. 

09:03:45 Preprocessing complete!


## Preview - Can Nodule Neurons be Isolated from Linker Neurons?

Here I visualise the initial output generated from HDBSCAN clustering on xyz coordinates on a small sample of the drosophila connectome. The aim is to ensure the clustering parameters assign cluster IDs to groups in a way that approximates real nodules. As the 3D plot shows, the Drosophila connectome cannot be segregated into 'nodules' using this approach. Indeed, most synapses are equidistant from each other in 3D space. To isolate 'nodule'-like structures and generate visualisations such as those seen in the literature, filtering by cell type is required, however, cell type annotations are not present in this dataset. Interestingly, there do appear to be some patches of yellow, blue, and pink, indicating there may still be some differentiation in neurotransmitter probabilities across neuropils. The remainder of this analysis will focus on comparing neurotransmitter probabilities across neuropils using the unclustered dataframe.

In [ ]:
connectome = pipeline.load_connectome("data/tiny.parquet")
connectome = pipeline.normalise_nt_probs(connectome)
connectome = pipeline.attach_synapse_coords(connectome)
condensed = pipeline.condense(connectome)
condensed = util.do_hdbscan(condensed, MINSIZE)
clustered = pipeline.extend(condensed, connectome)

09:04:24 Loading connectome ...
09:04:25 Connectome loaded
09:04:25 Normalising neurotransmitter probabilities ...
09:04:25 Neurotransmitter probabilities normalised
09:04:25 Attaching coordinates ...


In [ ]:
# Plot 500,000 points
plotter = brainz.get_plotter(clustered, "hdbscan_id")
brainz.save(plotter, OUTDIR, _id="clusterd_brain_map")
print("Saved brain map!")
plotter.show()

In [ ]:
plotter.close() # Free memory

## Visualise Neurotransmitter Probability Distributions

### Overall Neurotransmitter Probability Distributions

In [ ]:
# Sample ~1 million points to speed up render (using matplotlib)
#sample = util.downsample(connectome, 1_000_000)
sample = connectome
filename = os.path.join(OUTDIR, "overall_distribution.svg")
plotting.plot_overall_distribution(sample, "Full Dataset", filename)

### Get Neuropil Summary Statistics

In [ ]:
neuropils = pipeline.get_neuropil_summary_stats(connectome)
neuropils.head(10)

### Neuropil Probability Distributions with respect to Size (Number of Synapses)

In [ ]:
filename = os.path.join(OUTDIR, "mean_neuropil_probs.svg")
plotting.plot_mean_nt_probs_by_neuropil_size(neuropils, filename)

### Neurotransmitter Probability Variance by Neuropil Size

In [ ]:
filename = os.path.join(OUTDIR, "neuropil_nt_prob_variance_by_size.svg")
plotting.plot_variance_by_neuropil_size(neuropils, filename)

### Neuropil Probability Distributions

In [ ]:
filename = os.path.join(OUTDIR, "neuropil_probability_distributions.svg")
plotting.plot_hex_per_neuropil(connectome, filename)

## Statistical Analysis

I want to know whether neurotransmitter probabilities are different between neuropils. Because probabilities are a form of compositional data (the sum of the variables is 1, and each variable is bounded between 0 and 1), the Dirichlet distribution is suitable. This statistical analysis is a simple one based on average synapse probabilities within neuropils. A more statistically robust method would involve using the 'other' column to calculate the ranges of excitatory and inhibitory probabilities for each synaptic connection, then running a statistical analysis that can handle comparing range values within a bounded interval, e.g., Bayesian Dirichlet regression with interval priors. The Dirichlet function I used can only operate on points, not ranges. There does not appear to be an out-of-box Dirichlet regression function in any Python libraries, so I will interface with the R DirichletReg package via rpy2.

In [ ]:
fitted, model_summary, null_summary, anova_result = do_stats.run(connectome)